# Splitting a class by its neighbourhood, then comparing firing

The reverse of the firing-based split. Here Wide units are labelled by the
company they keep on the array, biphasic neighbours against triphasic and
positive neighbours, and the two groups are then compared on firing statistics.

**Why this direction is worth trying.** Neighbourhood and firing are independent
measurements, so a firing difference between neighbourhood-defined groups is
genuine evidence. It also uses the structure already established: the clique
analysis found that biphasic and triphasic classes occupy partly separate
territory, with Wide spanning both. If Wide units sitting in biphasic territory
fire differently from those in triphasic territory, that is consistent with Wide
being a laminar mixture rather than one population.

**Three confounds, each controlled below.**

Base rates differ. MedBI is far more abundant than MedTRI, so a
biphasic-dominant neighbourhood arises by chance more often. Section 3 defines
the label relative to the expected composition rather than by raw counts.

Neighbour counts vary. A unit with two neighbours gets a noisy label. A minimum
is imposed and the sensitivity to it is checked.

Both neighbourhood composition and firing vary across arrays. The permutation
null in section 5 therefore shuffles within array, so an array where Wide units
happen to be both biphasic-adjacent and fast cannot by itself produce an
effect.

In [ ]:
import os
os.chdir('/CSNG/studekat/ripple_paper_clean_copy/code_new_filter')

In [ ]:
from functions_analysis import *
import pandas as pd, numpy as np, yaml, pickle, itertools
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [ ]:
with open("/CSNG/studekat/ripple_paper_clean_copy/code_new_filter/params_analysis.yml") as f:
    params_analysis = yaml.safe_load(f)

DF_FOLDER = '/CSNG/studekat/ripple_paper_clean_copy/dataframes_new_filter'
TYPE_REC = 'NATIM'                 # 'NATIM' or 'RS'
MONKEY_LIST = ['N','F'] if TYPE_REC == 'NATIM' else ['L','N','F']
PROP_FOLDER = 'sua_prop_all_NATIM' if TYPE_REC == 'NATIM' else 'sua_prop_all'
AREA = 'V12'
FINAL_CLASSES = params_analysis['final_classes']
CLASS_COLORS = params_analysis['colors_class']
CLASS_DICT = {'DOWN_narrow_shallow':'NarrBI','DOWN_narrow_sharp':'NarrTRI',
              'DOWN_wide':'Wide','DOWN_medium_shallow':'MedBI',
              'DOWN_medium_sharp':'MedTRI','UP':'Pos'}
KEY = ['monkey','date','array','cell_name']

# ---- THE DESIGN ----
TARGET = 'DOWN_wide'               # class whose units get labelled
GROUP_A = ['DOWN_narrow_shallow', 'DOWN_medium_shallow']              # biphasic
GROUP_B = ['DOWN_narrow_sharp', 'DOWN_medium_sharp', 'UP']            # triphasic + positive
A_NAME, B_NAME = 'BI', 'TRI'
# Wide is deliberately excluded from both groups: it is the class being
# labelled, so counting other Wide units as neighbours would mix the question.

QUALITY_LEVEL = 'pass_k3'
MIN_NEIGHBOURS = 3                 # below this the label is too noisy
N_PERM = 1000
ALPHA = 0.05

## 1. Load

In [ ]:
def load_pkls(folder, monkeys, type_rec, tag=True):
    dfs = []
    for m in monkeys:
        for date in params_analysis['dates'][m][type_rec]:
            p = f'{DF_FOLDER}/{folder}/monkey{m}_all_arrays_date_{date}.pkl'
            try:
                with open(p,'rb') as f: d = pickle.load(f)
                if tag: d['monkey'] = m; d['date'] = date
                dfs.append(d)
            except Exception:
                print(f'   missing {folder}: {m} {date}')
    return pd.concat(dfs, ignore_index=True) if dfs else None

df_wf = load_pkls(PROP_FOLDER, MONKEY_LIST, TYPE_REC)
df_wf['area_merged'] = [a if a in ['V4','IT'] else 'V12' for a in df_wf['area']]
df_wf = df_wf[~df_wf['ch_is_noisy_100Hz'] & ~df_wf['ch_is_noisy_120Hz']]
df_wf = df_wf[df_wf['area_merged'] == AREA].reset_index(drop=True)

fire_folder = 'sua_firing_NATIM' if TYPE_REC == 'NATIM' else None
df_fire = load_pkls(fire_folder, MONKEY_LIST, TYPE_REC, tag=False) if fire_folder else None

with open(f'{DF_FOLDER}/sua_quality_{TYPE_REC}/unit_inclusion_list.pkl','rb') as f:
    df_qual = pickle.load(f)

df = df_wf.copy()
if df_fire is not None:
    fcols = [c for c in df_fire.columns if c not in df_wf.columns or c in KEY]
    df = df.merge(df_fire[fcols], on=KEY, how='left')

qcols = KEY + ['n_quality_pass'] + [c for c in df_qual.columns if c.startswith('pass_')]
qcols += [c for c in ['viol_ratio_min','coinc_ratio_worst','wf_shape_pc1_modesep']
          if c in df_qual.columns]
qcols = [c for c in dict.fromkeys(qcols) if c in df_qual.columns]
df = df.merge(df_qual[qcols], on=KEY, how='left')
df[QUALITY_LEVEL] = df[QUALITY_LEVEL].fillna(False).astype(bool)

d_ok = df[df[QUALITY_LEVEL] & (df['channel_order'] > -1)].copy()
print(f'{df.shape[0]} units, {d_ok.shape[0]} passing {QUALITY_LEVEL}')
print()
print(d_ok['final_class'].value_counts().rename(index=CLASS_DICT).to_string())

## 2. Count each unit's neighbours by class

Neighbours are electrodes differing by less than two positions in row and
column, matching the main analysis, so same-electrode neighbours count. Counting
is per array and date, since units recorded on different arrays or days were
never simultaneously present.

In [ ]:
def layout_coords(layout):
    c = np.full((64,2), -1.0)
    for ch in range(64):
        idx = np.where(layout == ch)
        if len(idx[0]): c[ch] = (idx[0][0], idx[1][0])
    return c

def neighbour_counts(d):
    """
    For every unit, the number of neighbours of each class, plus the block it
    belongs to. Returns the input frame with count columns appended.
    """
    lay = {}
    for m in d['monkey'].unique():
        for par in ['odd','even']:
            lay[(m,par)] = layout_coords(
                np.array(params_analysis['layout'][f'{m}_{par}']))
    v_areas = {m: [a in ['V1','V2'] for a in params_analysis['areas'][m]]
               for m in d['monkey'].unique()}

    out = []
    for (mk, date, array), g in d.groupby(['monkey','date','array'], sort=False):
        array = int(array)
        if not (1 <= array <= 16) or not v_areas[mk][array-1]:
            continue
        coords = lay[(mk, 'even' if array % 2 == 0 else 'odd')]
        ch = g['channel_order'].values.astype(int)
        ok = (ch >= 0) & (ch < 64)
        g = g[ok].copy(); ch = ch[ok]
        xy = coords[ch]
        good = xy[:,0] >= 0
        g = g[good].copy(); xy = xy[good]
        n = len(g)
        if n < 2:
            continue
        adj = ((np.abs(xy[:,None,0] - xy[None,:,0]) < 2) &
               (np.abs(xy[:,None,1] - xy[None,:,1]) < 2))
        np.fill_diagonal(adj, False)
        cls = g['final_class'].values
        for cl in FINAL_CLASSES:
            g[f'nb_{CLASS_DICT[cl]}'] = adj[:, cls == cl].sum(axis=1)
        g['nb_total'] = adj.sum(axis=1)
        g['block'] = f'{mk}_{date}_{array}'
        out.append(g)
    return pd.concat(out, ignore_index=True)

d_nb = neighbour_counts(d_ok)
NB_COLS = [f'nb_{CLASS_DICT[c]}' for c in FINAL_CLASSES]
print(f'{d_nb.shape[0]} units with neighbour counts, {d_nb["block"].nunique()} blocks')
print()
print('median neighbours per unit:', int(d_nb['nb_total'].median()))
print(d_nb[NB_COLS + ['nb_total']].describe(percentiles=[.25,.5,.75]).round(2).to_string())

## 3. Label the target units by neighbourhood

Raw counts would be misleading, because the two groups differ in abundance. The
label is therefore defined by comparing each unit's neighbourhood against the
composition expected from the class frequencies in the same block. A unit is
called `BI` when the biphasic share of its classified neighbours exceeds the
block expectation, and `TRI` when it falls below.

In [ ]:
A_COLS = [f'nb_{CLASS_DICT[c]}' for c in GROUP_A]
B_COLS = [f'nb_{CLASS_DICT[c]}' for c in GROUP_B]

d_nb['n_A'] = d_nb[A_COLS].sum(axis=1)
d_nb['n_B'] = d_nb[B_COLS].sum(axis=1)
d_nb['n_AB'] = d_nb['n_A'] + d_nb['n_B']

# expected biphasic share, per block, from the class composition of that block
exp_frac = {}
for blk, g in d_nb.groupby('block'):
    nA = g['final_class'].isin(GROUP_A).sum()
    nB = g['final_class'].isin(GROUP_B).sum()
    exp_frac[blk] = nA / (nA + nB) if (nA + nB) > 0 else np.nan
d_nb['exp_A_frac'] = d_nb['block'].map(exp_frac)
d_nb['obs_A_frac'] = d_nb['n_A'] / d_nb['n_AB'].replace(0, np.nan)
d_nb['A_excess'] = d_nb['obs_A_frac'] - d_nb['exp_A_frac']

tgt = d_nb[d_nb['final_class'] == TARGET].copy()
tgt = tgt[tgt['n_AB'] >= MIN_NEIGHBOURS]
tgt['nb_group'] = np.where(tgt['A_excess'] > 0, A_NAME,
                    np.where(tgt['A_excess'] < 0, B_NAME, 'tied'))
tgt = tgt[tgt['nb_group'] != 'tied']

GROUPS = [A_NAME, B_NAME]
G_COLORS = {A_NAME: 'darkorange', B_NAME: 'royalblue'}

print(f'{CLASS_DICT[TARGET]} units with >= {MIN_NEIGHBOURS} classified neighbours: '
      f'{tgt.shape[0]}')
print(tgt['nb_group'].value_counts().to_string())
print()
print(f'expected biphasic share across blocks: '
      f'{np.nanmedian(list(exp_frac.values())):.3f}')
print(tgt.groupby('nb_group')[['n_A','n_B','n_AB','obs_A_frac','A_excess']]
        .median().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), dpi=120)
axes[0].hist(tgt['obs_A_frac'], bins=25, color='gray', alpha=0.85)
axes[0].axvline(np.nanmedian(list(exp_frac.values())), color='r', ls='--',
                label='block expectation')
axes[0].set_xlabel('observed biphasic share of neighbours')
axes[0].legend(fontsize=8, frameon=False)

axes[1].hist(tgt['A_excess'], bins=30, color='gray', alpha=0.85)
axes[1].axvline(0, color='r', ls='--')
axes[1].set_xlabel('excess over block expectation')

for g in GROUPS:
    axes[2].hist(tgt.loc[tgt['nb_group']==g, 'nb_total'], bins=20, alpha=0.6,
                 color=G_COLORS[g], label=g)
axes[2].set_xlabel('total neighbours'); axes[2].legend(fontsize=8, frameon=False)
for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 4. Checks before comparing firing

Three things could produce a spurious firing difference: the groups differing in
sorting quality, in how many neighbours they have, or in which animal or array
they come from.

In [ ]:
qm = [c for c in ['viol_ratio_min','coinc_ratio_worst','wf_shape_pc1_modesep',
                  'nb_total','n_spikes_total'] if c in tgt.columns]
print('these should NOT differ between groups:')
print(tgt.groupby('nb_group')[qm].median().round(3).to_string())
print()
for c in qm:
    a = tgt.loc[tgt['nb_group']==A_NAME, c].dropna()
    b = tgt.loc[tgt['nb_group']==B_NAME, c].dropna()
    if len(a) > 10 and len(b) > 10:
        u = stats.mannwhitneyu(a, b)
        rb = 2*u.statistic/(len(a)*len(b)) - 1
        flag = '  <-- CONCERN' if abs(rb) > 0.2 else ''
        print(f'   {c:24s} rank-biserial {rb:+.3f}  p={u.pvalue:.2e}{flag}')
print()
ct = pd.crosstab(tgt['nb_group'], tgt['monkey'])
print('by animal:'); print(ct.to_string())
chi2, p, _, _ = stats.chi2_contingency(ct)
print(f'   chi2={chi2:.1f}, p={p:.2e}')
print()
print('blocks contributing to each group:')
print(tgt.groupby('nb_group')['block'].nunique().to_string())

## 5. Firing comparison, with a within-array null

The observed difference is compared against a null in which the group labels are
shuffled among target units **within each block**. That holds constant any
array-level property, so an array whose Wide units happen to be both
biphasic-adjacent and fast cannot generate an effect on its own.

In [ ]:
FIRE_METRICS = ['FR_baseline','FR_transient','FR_peak_evoked','modulation_index',
                'LV_evoked','LV_baseline','CV2_evoked','CV_ISI_evoked',
                'burst_index_evoked','frac_spikes_in_burst_evoked',
                'spikes_per_burst_evoked','psth_decay_ratio',
                'first_spike_latency_s','first_spike_jitter_s']
FIRE_METRICS = [m for m in FIRE_METRICS if m in tgt.columns]
if not FIRE_METRICS:
    FIRE_METRICS = [m for m in ['FR','CV_ISI','norm_RB_phase_selectivity_spikes']
                    if m in tgt.columns]
print('metrics compared:', FIRE_METRICS)

def obs_diff(d, metric):
    a = d.loc[d['nb_group']==A_NAME, metric]
    b = d.loc[d['nb_group']==B_NAME, metric]
    a, b = a.dropna(), b.dropna()
    if len(a) < 10 or len(b) < 10:
        return np.nan, np.nan, len(a), len(b)
    u = stats.mannwhitneyu(a, b)
    return (np.median(a) - np.median(b),
            2*u.statistic/(len(a)*len(b)) - 1, len(a), len(b))

rng = np.random.default_rng(0)
blocks = tgt['block'].values
lab = tgt['nb_group'].values
uniq_blocks = np.unique(blocks)
block_idx = {b: np.where(blocks == b)[0] for b in uniq_blocks}

rows = []
for metric in FIRE_METRICS:
    d_obs, rb_obs, na, nb_ = obs_diff(tgt, metric)
    if not np.isfinite(rb_obs):
        continue
    vals = tgt[metric].values
    null = np.empty(N_PERM)
    lab_p = lab.copy()
    for p in range(N_PERM):
        for b in uniq_blocks:
            i = block_idx[b]
            if len(i) > 1:
                lab_p[i] = rng.permutation(lab[i])
        a = vals[(lab_p == A_NAME)]; bb = vals[(lab_p == B_NAME)]
        a = a[np.isfinite(a)]; bb = bb[np.isfinite(bb)]
        if len(a) < 10 or len(bb) < 10:
            null[p] = np.nan; continue
        u = stats.mannwhitneyu(a, bb)
        null[p] = 2*u.statistic/(len(a)*len(bb)) - 1
    nn = null[np.isfinite(null)]
    p_two = (2*min((nn >= rb_obs).sum()+1, (nn <= rb_obs).sum()+1)/(len(nn)+1)
             if len(nn) else np.nan)
    rows.append({'metric': metric, f'median_{A_NAME}': round(
                    tgt.loc[tgt['nb_group']==A_NAME, metric].median(), 4),
                 f'median_{B_NAME}': round(
                    tgt.loc[tgt['nb_group']==B_NAME, metric].median(), 4),
                 'rank_biserial': round(rb_obs, 3),
                 'null_mean': round(float(np.nanmean(nn)), 3),
                 'null_sd': round(float(np.nanstd(nn)), 3),
                 'z_vs_null': round((rb_obs - np.nanmean(nn))/np.nanstd(nn), 2)
                             if np.nanstd(nn) > 0 else np.nan,
                 'p_perm': round(min(p_two, 1.0), 4),
                 f'n_{A_NAME}': na, f'n_{B_NAME}': nb_})
df_fire_cmp = pd.DataFrame(rows)
df_fire_cmp['p_holm'] = multipletests(df_fire_cmp['p_perm'], method='holm')[1]
df_fire_cmp = df_fire_cmp.reindex(
    df_fire_cmp['z_vs_null'].abs().sort_values(ascending=False).index)
print()
print(df_fire_cmp.to_string(index=False))
print()
print('rank_biserial is the raw effect; z_vs_null is the same effect measured')
print('against the within-block null, and is the quantity to interpret.')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5), dpi=120)
d_plot = df_fire_cmp.dropna(subset=['z_vs_null'])
cols = ['maroon' if s else 'lightgray' for s in (d_plot['p_holm'] < ALPHA)]
ax.barh(d_plot['metric'], d_plot['z_vs_null'], color=cols, alpha=0.9)
ax.axvline(0, color='k', lw=1)
for xv in (-2, 2): ax.axvline(xv, color='r', ls=':', lw=1)
ax.set_xlabel(f'z of the {A_NAME} vs {B_NAME} effect, against the within-block null')
ax.set_title(f'{CLASS_DICT[TARGET]} units grouped by neighbourhood')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

show = d_plot['metric'].tolist()[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7), dpi=120)
for ax, m in zip(axes.flat, show):
    sns.violinplot(data=tgt, x='nb_group', y=m, hue='nb_group', order=GROUPS,
                   palette=G_COLORS, inner='box', cut=0, ax=ax, legend=False)
    for v in ax.collections: v.set_alpha(0.7)
    if m.startswith('FR_') or m == 'FR': ax.set_yscale('log')
    ax.set_xlabel('')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for ax in axes.flat[len(show):]: ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Where do the groups sit relative to the other classes?

If the neighbourhood split is capturing something real, the group sitting among
biphasic units should resemble the biphasic classes on the metrics that
distinguish them, and likewise for the triphasic side.

In [ ]:
top = df_fire_cmp.dropna(subset=['z_vs_null'])['metric'].tolist()[:3]
fig, axes = plt.subplots(1, len(top), figsize=(5*len(top), 4), dpi=120)
if len(top) == 1: axes = [axes]
for ax, m in zip(axes, top):
    parts = []
    for cl in FINAL_CLASSES:
        if cl == TARGET: continue
        v = d_ok.loc[d_ok['final_class']==cl, m].dropna()
        if len(v) > 20:
            parts.append((CLASS_DICT[cl], v, CLASS_COLORS[cl]))
    for g in GROUPS:
        parts.append((f'W|{g}', tgt.loc[tgt['nb_group']==g, m].dropna(), G_COLORS[g]))
    bp = ax.boxplot([p[1] for p in parts], labels=[p[0] for p in parts],
                    patch_artist=True, showfliers=False)
    for patch, p in zip(bp['boxes'], parts):
        patch.set_facecolor(p[2]); patch.set_alpha(0.75)
    ax.set_ylabel(m, fontsize=9); ax.tick_params(labelsize=7, rotation=45)
    if m.startswith('FR_') or m == 'FR': ax.set_yscale('log')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Sensitivity to the neighbour threshold

The minimum number of classified neighbours trades label reliability against
sample size. The effect should be stable across reasonable values.

In [ ]:
rows = []
for mn in [2, 3, 4, 5, 6, 8]:
    t = d_nb[(d_nb['final_class']==TARGET) & (d_nb['n_AB'] >= mn)].copy()
    t['nb_group'] = np.where(t['A_excess'] > 0, A_NAME,
                      np.where(t['A_excess'] < 0, B_NAME, 'tied'))
    t = t[t['nb_group'] != 'tied']
    r = {'min_nb': mn, 'n_units': len(t),
         f'n_{A_NAME}': int((t['nb_group']==A_NAME).sum()),
         f'n_{B_NAME}': int((t['nb_group']==B_NAME).sum())}
    for m in df_fire_cmp.dropna(subset=['z_vs_null'])['metric'].tolist()[:3]:
        _, rb, na, nb_ = obs_diff(t, m)
        r[m] = round(rb, 3) if np.isfinite(rb) else np.nan
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
print()
print('Effects that change sign or vanish across thresholds should not be')
print('reported. A shrinking effect at high thresholds is expected from the')
print('smaller sample and is not by itself a problem.')

In [ ]:
# match on neighbour count, then repeat the key comparison
rng_m = np.random.default_rng(0)
keep = []
for n_nb, g in tgt.groupby('nb_total'):
    a = g[g['nb_group']==A_NAME].index.values
    b = g[g['nb_group']==B_NAME].index.values
    k = min(len(a), len(b))
    if k == 0: continue
    keep += list(rng_m.choice(a, k, replace=False))
    keep += list(rng_m.choice(b, k, replace=False))
t_m = tgt.loc[keep]
print(f'matched: {len(t_m)} units')
print(t_m.groupby('nb_group')['nb_total'].median().to_string())
for m in ['CV2_evoked','LV_evoked','modulation_index']:
    _, rb, na, nb_ = obs_diff(t_m, m)
    print(f'   {m:20s} rank-biserial {rb:+.3f}  (was {df_fire_cmp.set_index("metric").loc[m,"rank_biserial"]:+.3f})')

## 8. The same split applied to other classes

Wide is the natural target because it spans both territories, but the same
labelling can be applied to any class. If every class shows the same firing
difference by neighbourhood, the effect is a property of cortical location
rather than of Wide.

In [ ]:
rows = []
for cl in FINAL_CLASSES:
    if cl in GROUP_A or cl in GROUP_B:
        continue_note = 'in a neighbour group'
    else:
        continue_note = ''
    t = d_nb[(d_nb['final_class']==cl) & (d_nb['n_AB'] >= MIN_NEIGHBOURS)].copy()
    if len(t) < 60:
        continue
    t['nb_group'] = np.where(t['A_excess'] > 0, A_NAME,
                      np.where(t['A_excess'] < 0, B_NAME, 'tied'))
    t = t[t['nb_group'] != 'tied']
    r = {'class': CLASS_DICT[cl], 'note': continue_note, 'n': len(t)}
    for m in df_fire_cmp.dropna(subset=['z_vs_null'])['metric'].tolist()[:3]:
        _, rb, na, nb_ = obs_diff(t, m)
        r[m] = round(rb, 3) if np.isfinite(rb) else np.nan
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
print()
print('Classes marked "in a neighbour group" are partly labelled by their own')
print('kind, so their values are not independent and are shown for context only.')

In [ ]:
# Specificity: label every class using only neighbours of OTHER classes, so no
# class is partly labelled by its own kind. This is what makes the comparison
# across classes independent.

def _rb(d, metric):
    a = d.loc[d['nb_group'] == A_NAME, metric].dropna()
    b = d.loc[d['nb_group'] == B_NAME, metric].dropna()
    if len(a) < 10 or len(b) < 10:
        return np.nan
    u = stats.mannwhitneyu(a, b)
    return 2 * u.statistic / (len(a) * len(b)) - 1
    
rows = []
for cl in FINAL_CLASSES:
    GA = [c for c in GROUP_A if c != cl]
    GB = [c for c in GROUP_B if c != cl]
    if not GA or not GB:
        continue

    a_cols = [f'nb_{CLASS_DICT[c]}' for c in GA]
    b_cols = [f'nb_{CLASS_DICT[c]}' for c in GB]

    # block expectation for this particular neighbour definition
    exp2 = {}
    for blk, g in d_nb.groupby('block'):
        nA = g['final_class'].isin(GA).sum()
        nB = g['final_class'].isin(GB).sum()
        exp2[blk] = nA / (nA + nB) if (nA + nB) > 0 else np.nan

    t = d_nb[d_nb['final_class'] == cl].copy()
    t['n_A2'] = t[a_cols].sum(axis=1)
    t['n_B2'] = t[b_cols].sum(axis=1)
    t['n_AB2'] = t['n_A2'] + t['n_B2']
    t = t[t['n_AB2'] >= MIN_NEIGHBOURS]
    if len(t) < 60:
        rows.append({'class': CLASS_DICT[cl], 'n': len(t)})
        continue

    t['exc2'] = t['n_A2'] / t['n_AB2'] - t['block'].map(exp2)
    t['nb_group'] = np.where(t['exc2'] > 0, A_NAME,
                      np.where(t['exc2'] < 0, B_NAME, 'tied'))
    t = t[t['nb_group'] != 'tied']

    r = {'class': CLASS_DICT[cl], 'n': len(t),
         f'n_{A_NAME}': int((t['nb_group'] == A_NAME).sum()),
         f'n_{B_NAME}': int((t['nb_group'] == B_NAME).sum())}
    for m in ['CV2_evoked', 'LV_evoked', 'modulation_index']:
        if m in t.columns:
            r[m] = round(_rb(t, m), 3)
    rows.append(r)

print(pd.DataFrame(rows).to_string(index=False))
print()
print('Each class is labelled only by neighbours of OTHER classes, so these are')
print('independent of one another. If Wide stands out, the effect is specific;')
print('if every class shows it, regularity varies across cortical territory.')

## 9. Export

In [ ]:
out_dir = f'{DF_FOLDER}/neighbourhood_split_{TYPE_REC}'
ensure_dir_exists(out_dir)
tgt[KEY + ['nb_group','n_A','n_B','n_AB','obs_A_frac','exp_A_frac','A_excess',
           'block']].to_pickle(f'{out_dir}/neighbourhood_labels.pkl')
df_fire_cmp.to_csv(f'{out_dir}/firing_by_neighbourhood.csv', index=False)
d_nb[KEY + NB_COLS + ['nb_total','block']].to_pickle(
    f'{out_dir}/neighbour_counts_all_units.pkl')
print('saved to', out_dir)
print(f'target {CLASS_DICT[TARGET]}, groups {A_NAME}={[CLASS_DICT[c] for c in GROUP_A]}, '
      f'{B_NAME}={[CLASS_DICT[c] for c in GROUP_B]}')
print(f'min neighbours {MIN_NEIGHBOURS}, {N_PERM} permutations, within-block null')

## tSNE comparing two groups

In [ ]:
# ---- tSNE of average waveforms, Wide split by neighbourhood ----
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

WF_COL = 'avg_wf_zscored'
assert WF_COL in tgt.columns, f'{WF_COL} not in the frame'

d_wf = tgt[tgt[WF_COL].notna()].copy()
W = np.vstack(d_wf[WF_COL].values)
y = d_wf['nb_group'].values
print(f'{W.shape[0]} Wide units, waveform length {W.shape[1]}')
print(pd.Series(y).value_counts().to_string())

# --- 1. embedding of the Wide subgroups alone ---
Wz = StandardScaler().fit_transform(W)
tsne = TSNE(n_components=2, random_state=0, init='pca',
            perplexity=min(30, max(5, len(Wz)//20)))
E = tsne.fit_transform(Wz)
P = PCA(n_components=2).fit_transform(Wz)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), dpi=120)

ax = axes[0]
for g in GROUPS:
    m = y == g
    ax.scatter(E[m, 0], E[m, 1], s=14, alpha=0.45, color=G_COLORS[g], label=g)
ax.set_xlabel('tSNE 1'); ax.set_ylabel('tSNE 2')
ax.set_title('Wide waveforms, coloured by neighbourhood')
ax.legend(frameon=False, fontsize=8, markerscale=1.6)

ax = axes[1]
for g in GROUPS:
    m = y == g
    ax.scatter(P[m, 0], P[m, 1], s=14, alpha=0.45, color=G_COLORS[g], label=g)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('PCA of the same waveforms')

# --- 2. the mean waveform per group, which the embedding cannot show ---
ax = axes[2]
for g in GROUPS:
    m = y == g
    mu = W[m].mean(axis=0)
    se = W[m].std(axis=0) / np.sqrt(m.sum())
    x = np.arange(len(mu)) * 33          # samples to microseconds
    ax.plot(x, mu, color=G_COLORS[g], lw=2, label=f'{g} (n={m.sum()})')
    ax.fill_between(x, mu - 1.96*se, mu + 1.96*se, color=G_COLORS[g], alpha=0.25)
ax.set_xlabel('Time [micro s]'); ax.set_ylabel('Amplitude [z-scored]')
ax.set_title('Mean waveform, 95% CI')
ax.legend(frameon=False, fontsize=8)

for a_ in axes:
    a_.spines['top'].set_visible(False); a_.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

# --- 3. are they actually separable? tSNE cannot answer this ---
clf = LogisticRegression(max_iter=2000, class_weight='balanced')
cv = StratifiedKFold(5, shuffle=True, random_state=0)
acc = cross_val_score(clf, Wz, y, cv=cv, scoring='balanced_accuracy')
# label-shuffled null, so chance is measured rather than assumed
rng_s = np.random.default_rng(0)
null = [cross_val_score(clf, Wz, rng_s.permutation(y), cv=cv,
                        scoring='balanced_accuracy').mean() for _ in range(20)]
print()
print(f'waveform -> neighbourhood group, balanced accuracy '
      f'{acc.mean():.3f} +/- {acc.std():.3f}')
print(f'shuffled-label null {np.mean(null):.3f} +/- {np.std(null):.3f}')
print('Accuracy near the null means the waveforms do not distinguish the groups,')
print('however the embedding looks.')

# --- 4. and on the waveform features already computed ---
for feat in ['width_wf', 'first_peak_height']:
    if feat not in d_wf.columns:
        continue
    a = d_wf.loc[d_wf['nb_group'] == GROUPS[0], feat].dropna()
    b = d_wf.loc[d_wf['nb_group'] == GROUPS[1], feat].dropna()
    if len(a) > 10 and len(b) > 10:
        u = stats.mannwhitneyu(a, b)
        rb = 2*u.statistic/(len(a)*len(b)) - 1
        print(f'{feat:20s} {GROUPS[0]} {np.median(a):.3f}  '
              f'{GROUPS[1]} {np.median(b):.3f}  rb {rb:+.3f}  p={u.pvalue:.2e}')

## E-I dispute

In [ ]:
# ---- Each Wide subgroup against reference classes ----
# NarrBI is the candidate fast-spiking / interneuron class; MedBI is the other
# putative layer 4 class and the excitatory candidate. Comparing each Wide
# subgroup to both shows which it resembles.
#
# CAVEAT: NarrBI was not established as fast-spiking in this dataset. It is more
# regular and less bursty than Wide, but also slower, later and more jittery,
# which is the wrong direction for an interneuron. Resembling NarrBI therefore
# does not establish inhibitory identity; it establishes resemblance to NarrBI.

ANCHORS = {'NarrBI': 'DOWN_narrow_shallow',   # FS / interneuron candidate
           'MedBI':  'DOWN_medium_shallow'}   # RS / excitatory candidate
CMP_METRICS = [m for m in ['LV_evoked','CV2_evoked','burst_index_evoked',
                           'frac_spikes_in_burst_evoked','FR_baseline',
                           'FR_transient','FR_peak_evoked','modulation_index',
                           'first_spike_latency_s','first_spike_jitter_s',
                           'psth_decay_ratio']
               if m in tgt.columns]

def _rb_between(a_vals, b_vals):
    a = pd.Series(a_vals).dropna(); b = pd.Series(b_vals).dropna()
    if len(a) < 10 or len(b) < 10:
        return np.nan
    u = stats.mannwhitneyu(a, b)
    return 2 * u.statistic / (len(a) * len(b)) - 1

rows = []
for g in GROUPS:
    sub = tgt[tgt['nb_group'] == g]
    for anch_name, anch_cls in ANCHORS.items():
        ref = d_ok[d_ok['final_class'] == anch_cls]
        for m in CMP_METRICS:
            rb = _rb_between(sub[m], ref[m])
            rows.append({'wide_group': g, 'anchor': anch_name, 'metric': m,
                         'median_wide': round(sub[m].median(), 4),
                         'median_anchor': round(ref[m].median(), 4),
                         'rb_vs_anchor': round(rb, 3) if np.isfinite(rb) else np.nan})
df_anch = pd.DataFrame(rows)

piv = df_anch.pivot_table(index='metric', columns=['wide_group','anchor'],
                          values='rb_vs_anchor')
print('rank-biserial of each Wide subgroup against each anchor')
print('(near 0 = indistinguishable from that anchor)')
print()
print(piv.round(3).to_string())

In [ ]:
# ---- Which anchor is each subgroup closer to, overall? ----
print('mean |rank-biserial| across metrics, smaller = more similar to that anchor')
print()
summ = (df_anch.dropna(subset=['rb_vs_anchor'])
        .assign(absrb=lambda d: d['rb_vs_anchor'].abs())
        .groupby(['wide_group','anchor'])['absrb']
        .agg(['mean','median','count']).round(3))
print(summ.to_string())
print()
for g in GROUPS:
    s = summ.loc[g]['mean']
    closer = s.idxmin()
    print(f'  Wide|{g} is closer to {closer} '
          f'({s.min():.3f} vs {s.max():.3f})')

# per-metric view, so a single dominant metric is visible rather than hidden
fig, axes = plt.subplots(1, len(GROUPS), figsize=(7*len(GROUPS), 5), dpi=120,
                         sharey=True)
if len(GROUPS) == 1: axes = [axes]
for ax, g in zip(axes, GROUPS):
    d_g = df_anch[df_anch['wide_group'] == g].dropna(subset=['rb_vs_anchor'])
    p = d_g.pivot(index='metric', columns='anchor', values='rb_vs_anchor')
    p.plot.barh(ax=ax, alpha=0.9,
                color={'NarrBI': CLASS_COLORS['DOWN_narrow_shallow'],
                       'MedBI': CLASS_COLORS['DOWN_medium_shallow']})
    ax.axvline(0, color='k', lw=1)
    ax.set_xlabel('rank-biserial vs anchor (0 = same as anchor)')
    ax.set_title(f'Wide | {g} neighbourhood')
    ax.legend(frameon=False, fontsize=8, title='anchor')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## How to read this

**Section 3 is where the base-rate problem is handled.** Labelling by raw counts
would call almost everything biphasic, because MedBI and NarrBI outnumber the
triphasic classes. The label is instead defined relative to the composition of
the same block, so it asks whether a unit has more biphasic neighbours than its
own array would predict.

**Section 4 is the gate.** If the groups differ in sorting quality or in
neighbour count, the firing difference may follow from that instead. Anything
above 0.2 rank-biserial is flagged.

**Section 5 is the result, and `z_vs_null` is the number to read**, not the raw
rank-biserial. Neighbourhood composition and firing both vary across arrays, so
a raw comparison pools within-array and between-array structure. The within-block
null removes the latter.

**Section 8 is the specificity check.** If every class shows the same
neighbourhood effect, it reflects cortical position generally rather than
anything about Wide. Note that classes belonging to a neighbour group are partly
labelled by their own kind and are shown for context only.

**On interpretation.** A firing difference between neighbourhood-defined groups
says the two territories differ, not what the territories are. Given that the
biphasic classes are putative layer 4 and the triphasic classes putative deep
layers, the most parsimonious reading is laminar: Wide units in layer 4
neighbourhoods fire like layer 4, and those in deep neighbourhoods fire like
deep layers. That is a claim about location, not about cell type.